In [ ]:
# Setup: add project root
import sys
import os
import logging

# Configure logging to display in notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

print("Imports OK")

In [ ]:
from src.context.context_factory import create_context
save_path = os.path.join(BASE,"data/results/single_paper_results/agronomyjournal.csv")
folder_source = os.path.join(BASE, 'data/paper_to_extract/Agronomy Journal - 2021 - Hegstad - Agronomic and efficacy evaluations of oxathiapiprolin as a soybean seed treatment_formatted.md')
data_context = create_context(
    source=folder_source,
    name='paper_to_extract'
)

In [ ]:
# Updated objective enforcing schema-aligned keys only
from src.standards import METADATA_STANDARDS
standard = METADATA_STANDARDS["climate_vs_cropyield"]

orchestrator = Orchestrator(topology_name="fast")

objective = f'''You are an expert agricultural data extraction specialist. Your task is to extract **measured crop yield information** and related agronomic/context variables from scientific research papers. You must reason carefully and follow a multi-step extraction process to ensure accurate and complete data.

**META-ANALYTIC SCHEMA (CRITICAL)**:
{standard}

**SCHEMA HANDLING RULES**
- The meta-analytic schema above is expressed as JSON and/or `field: description` pairs.
- Treat the JSON keys / the text before the colon as the exact field names to use as keys in every yield record (e.g., `crop_type`, `yield_value`, `location`, `year`, `Treatment`, `Tillage`, `Soil_property`, `climate`, `remote_sensing_data`).
- When you construct any meta-analytic records (proposed, refined, or final), you MUST use these field names as the keys.
- The natural-language descriptions in the schema are only guidance and must **never** be copied as record values; every value must be the concrete value extracted from the paper.
- Do not introduce new top-level keys or rename existing ones in the final `yield_records`; use only the keys that appear in the schema.

**EXTRACTION PROCESS**

**Step 1: Label the Original Document**
The first step is to label the original document text by adding XML tags around schema-relevant values. This should be done by the labeller agent.

The labeller must read the complete original document from start to finish and add XML tags around schema-relevant text spans using schema field names (e.g., <crop_type>winter wheat</crop_type>, <yield_value>529.58 g/m2</yield_value>). The labeller must preserve ALL original text, formatting, paragraphs, and structure exactly as it appears. The labeller must NOT create summaries, lists, tables, or structured formats - only add tags to the original text. The output should be the complete original document with tags inserted inline, nothing else.

**Step 2: Extract and Structure Records**
After the document is labeled, extract structured records from the labeled text. For each identified yield anchor, extract all associated schema fields from the surrounding context (paragraphs, section headers, tables, captions). The schema fields to extract are:
- crop_type: Specific crop name (e.g., maize, wheat, rice, soybean, corn, etc.)
- yield_value: Actual yield measurement with units (e.g., 8.5 t/ha, 3500 kg/ha, 42 bu/acre)
- location: Study location (prefer coordinates if reported; otherwise country/province/state/city/research station/farm name)
- year: Data collection year or growing season (e.g., 2018, 2019–2020; not publication year)
- Treatment: The experimental treatment (e.g., Irrigation amount/frequency or total nitrogen fertilizer application amount and frequency)
- Tillage: Density and Planting (e.g., Row/plant spacing, density, etc.)
- Soil_property: Soil properties of the experimental sites (e.g., Texture, organic matter, pH, electrical conductivity, total nitrogen, available phosphorus/potassium, bulk density, field water holding capacity, available water capacity, and soil depth)
- climate: Climate data (e.g., daily/decadal Tmax/Tmin/Tmean, precipitation, radiation/sunshine, wind speed, VPD, ET0, GDD, KDD, cumulative precipitation, critical windows)
- remote_sensing_data: Raw spectral data or vegetation index (e.g., NDVI, EVI, GNDVI, NDRE, WDRVI, MCARI, red edge index, etc.)

Scan the entire content (including paragraphs, tables, captions, footnotes, and supplements) to locate all sentences or table entries that mention **actual measured crop yield values** (e.g., "maize yield was 7.2 t/ha"). Ignore model performance metrics (e.g., RMSE, R²), yield gaps, and simulated/modelled outputs.

**Complete Missing Fields and Validate**
If any required field for a yield record is **missing**, search specific related sections to retrieve it:
- For missing **location** or **year**, check **Materials and Methods / Site description**.
- For missing **crop type**, check **Abstract** and **Materials and Methods**.
- For missing **treatment (water/fertilizer)**, check **Field management / Experimental design / Plot / Block / Treatment / Table**.
- For missing **tillage (density & cultivation)**, check **Planting / Agronomic practices / Cultivation / Management**.
- For missing **soil properties**, check **Soil / Site description / Materials and Methods / Table**.
- For missing **climate**, check **Climate / Weather / Meteorology / Data sources**.
- For missing **UAV data**, check **UAV / UAS / Flight / Sensor / Data acquisition / Table / Figure captions**.

If you cannot find a field with **reasonable certainty**, discard the entire record. Only keep records where **all required fields are present and consistent**.

**Structure and Finalize Records**
For each complete and validated record, construct a structured JSON entry following the schema above. Keep original units and expressions; **do not convert**. Put the ≤30-word evidence snippet for the **yield** in `"yield_notes"`. If an associated field is not found with reasonable certainty, set its value to `null` but still populate its source_section, confidence, and notes when applicable. Apply de-duplication across records where (crop_type, yield_value+unit, location, year, treatment if any) are identical. Ensure all records strictly conform to the schema field names and structure.

**TARGET DATA TO EXTRACT**:
- ✅ INCLUDE: Actual field-measured or reported yields (e.g., from experiments, harvest trials)
- ❌ EXCLUDE: Model evaluation metrics (RMSE, R², MAE), yield gaps, predictions, correlation coefficients

**OUTPUT FORMAT**: Return results in JSON format with an array of yield records where each record's keys are **exactly** the field names from the schema and the values are the extracted data. If no yield data is found, return an empty array. Return ONLY the JSON response, no additional text or explanations.'''

plan = orchestrator.generate_plan(context=data_context, objective=objective)


In [ ]:
for i, step in enumerate(plan.steps):
    print("=" * 50)
    print(f"Step {i}: {step.task}")
    print(f"  Player: {step.player}")
    print(f"  Rationale: {step.rationale}")
    print(f"  Required Artifacts: {step.inputs}")
    print(f"  Produced Artifacts: {step.outputs}")
    # Also show execution context information for this run
    print(f"  ExecutionContext: name={data_context.name}, type={data_context.context_type.value}")
    print(f"  Resources: {data_context.resources}")

In [ ]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

In [ ]:
result = orchestrator.execute_plan(plan=plan, context=data_context, objective=objective)

In [ ]:
# Inspect execution results
if 'result' in dir() and result:
    print("Execution Results Summary:")
    print("=" * 50)
    print(f"Success: {result.success}")
    print(f"Steps Completed: {result.steps_completed}/{result.plan_steps_count}")
    
    print("\n--- Step Results ---")
    for step_result in result.step_results:
        print(f"\nStep {step_result.step_index + 1}: {step_result.task}")
        print(f"  Player Role: {step_result.player_role}")
        print(f"  Success: {step_result.success}")
        print(f"  Debate Rounds: {step_result.debate_rounds_completed}")
        print(f"  Artifacts Produced: {list(step_result.artifacts.keys())}")
        if step_result.error:
            print(f"  Error: {step_result.error}")
else:
    print("No results to inspect. Run the execution cell first.")

In [ ]:
from pprint import pprint
# View final workspace and metadata
if 'result' in dir() and result:
    print("Final Workspace Artifacts:")
    print("=" * 50)
    for name, value in result.final_workspace.items():
        print(f"\n--- {name} ---")
        # Truncate long values for display
        value_str = str(value)
        if len(value_str) > 500:
            print(value_str[:500] + "...")
        else:
            print(value_str)
    
    print("\n" + "=" * 50)
    print("Final Outputs:")
    print("=" * 50)
    pprint(result.final_output)
else:
    print("No results to display.")

In [ ]:
from src.utils import print_json_records

# Example usage:
if 'result' in dir() and result:
    if "final_meta_analysis_records" in result.final_workspace:
        print_json_records(result.final_workspace["final_meta_analysis_records"])
    else:
        print("No 'final_meta_analysis_records' found in workspace.")
        print("Available keys:", list(result.final_workspace.keys()))
else:
    print("No results available.")

In [ ]:
import csv
import json
import os
import re

def _normalize_cell_value(value):
    """
    Normalize a Python value into a CSV cell string.
    """
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        # Store complex structures as JSON strings
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def save_json_records_to_csv(json_string, path, record_key="yield_records"):
    """
    Parse JSON records and save them to a CSV file.

    Behavior:
    - If the file does not exist or is empty, write header + rows.
    - If the file exists and is non-empty, verify that the header (column names)
      matches the new records' fields. If they match, append rows. If not, raise
      a ValueError with details about the mismatch.

    Args:
        json_string: String containing JSON (may include markdown code blocks), a dict, or a list.
        path: Path to the CSV file (str or Path-like). This is required.
        record_key: Key in the JSON object that contains the array of records.

    Returns:
        The number of records written.

    Raises:
        ValueError: If the existing CSV header does not match the new records' schema.
    """
    # Handle list input directly
    if isinstance(json_string, list):
        records = json_string
    # Handle dict input - extract records using record_key or convert to JSON string
    elif isinstance(json_string, dict):
        # Try to extract records from dict using record_key
        if record_key in json_string:
            value = json_string[record_key]
            # If it's already a list, use it directly
            if isinstance(value, list):
                records = value
            # If it's a string, try to parse it as JSON
            elif isinstance(value, str):
                # Strip markdown code blocks if present
                cleaned = value.strip()
                if cleaned.startswith("```json"):
                    cleaned = re.sub(r'^```json\s*', "", cleaned)
                if cleaned.startswith("```"):
                    cleaned = re.sub(r'^```\s*', "", cleaned)
                if cleaned.endswith("```"):
                    cleaned = re.sub(r'\s*```$', "", cleaned)
                cleaned = cleaned.strip()
                
                # Parse JSON
                parsed = json.loads(cleaned)
                if isinstance(parsed, list):
                    records = parsed
                elif isinstance(parsed, dict) and "records" in parsed and isinstance(parsed["records"], list):
                    records = parsed["records"]
                else:
                    records = [parsed] if isinstance(parsed, dict) else []
            else:
                records = []
        else:
            # Try common keys
            for key in ["records", "yield_records", "data", "final_meta_analysis_records"]:
                if key in json_string:
                    value = json_string[key]
                    if isinstance(value, list):
                        records = value
                        break
                    elif isinstance(value, str):
                        # Parse JSON string
                        cleaned = value.strip()
                        if cleaned.startswith("```json"):
                            cleaned = re.sub(r'^```json\s*', "", cleaned)
                        if cleaned.startswith("```"):
                            cleaned = re.sub(r'^```\s*', "", cleaned)
                        if cleaned.endswith("```"):
                            cleaned = re.sub(r'\s*```$', "", cleaned)
                        cleaned = cleaned.strip()
                        parsed = json.loads(cleaned)
                        if isinstance(parsed, list):
                            records = parsed
                            break
                        elif isinstance(parsed, dict) and "records" in parsed:
                            records = parsed["records"]
                            break
            else:
                # If no array found, treat the dict itself as a single record
                records = [json_string]
    else:
        # Handle string input (same logic as print_json_records)
        # Strip markdown code blocks if present
        cleaned = json_string.strip()
        if cleaned.startswith("```json"):
            cleaned = re.sub(r'^```json\s*', "", cleaned)
        if cleaned.startswith("```"):
            cleaned = re.sub(r'^```\s*', "", cleaned)
        if cleaned.endswith("```"):
            cleaned = re.sub(r'\s*```$', "", cleaned)
        cleaned = cleaned.strip()

        # Parse JSON and extract records
        data = json.loads(cleaned)

        if isinstance(data, list):
            records = data
        elif isinstance(data, dict):
            # Try common keys
            for key in [record_key, "records", "yield_records", "data", "final_meta_analysis_records"]:
                if key in data and isinstance(data[key], list):
                    records = data[key]
                    break
            else:
                # If no array found, treat the dict itself as a single record
                records = [data]
        else:
            raise ValueError(f"Unexpected JSON structure: {type(data)}")

    if not records:
        # Nothing to write
        return 0

    # Determine fieldnames from union of all keys
    fieldnames = sorted({key for rec in records for key in rec.keys()})

    csv_path = str(path)
    file_exists = os.path.exists(csv_path)
    file_empty = (not file_exists) or os.path.getsize(csv_path) == 0

    if file_empty:
        # New file or empty file: write header and rows
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for rec in records:
                row = {fn: _normalize_cell_value(rec.get(fn)) for fn in fieldnames}
                writer.writerow(row)
        return len(records)

    # Existing non-empty file: check header compatibility
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            existing_header = next(reader)
        except StopIteration:
            # File has no rows but is non-empty (corrupt or custom) - treat as mismatch
            existing_header = []

    if existing_header != fieldnames:
        existing_set = set(existing_header)
        new_set = set(fieldnames)
        only_in_existing = sorted(existing_set - new_set)
        only_in_new = sorted(new_set - existing_set)
        raise ValueError(
            "CSV schema mismatch between existing file and new records.\n"
            f"Existing header: {existing_header}\n"
            f"New header: {fieldnames}\n"
            f"Columns only in existing file: {only_in_existing}\n"
            f"Columns only in new records: {only_in_new}"
        )

    # Schemas match: append rows
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=existing_header)
        for rec in records:
            row = {fn: _normalize_cell_value(rec.get(fn)) for fn in existing_header}
            writer.writerow(row)

    return len(records)

save_json_records_to_csv(result.final_workspace, save_path, record_key="final_meta_analysis_records")

In [ ]:
result.final_workspace["final_meta_analysis_records"]